In [1]:
import numpy as np
import pandas as pd

### Day 1 — Bird Diversity & Habitat Analysis

In [2]:
np.random.seed(42)

sites = ["East Kolkata Wetlands", "Sundarban Fringe", "Nalban", "Santragachi"]
species = ["Egret", "Kingfisher", "Heron", "Cormorant", "Ibis", "Pied Hornbill"]

dates = pd.date_range("2025-01-01", "2025-06-01", freq="MS")

rows = []

for date in dates:
    for site in sites:
        for sp in species:
            rows.append({
                "date": date,
                "site": site,
                "species": sp,
                "count": np.random.poisson(
                    np.random.uniform(5, 30)
                ),
                "observation_hours": np.random.uniform(2, 8)
            })

birds = pd.DataFrame(rows)

# Introduce some missing observations
missing_idx = np.random.choice(
    birds.index,
    size=12,
    replace=False
)

birds.loc[missing_idx, "count"] = np.nan

birds.head()

,date,site,species,count,observation_hours
0,2025-01-01,East Kolkata Wetlands,Egret,15.0,2.935967
1,2025-01-01,East Kolkata Wetlands,Kingfisher,6.0,3.090950
2,2025-01-01,East Kolkata Wetlands,Heron,9.0,3.198043
3,2025-01-01,East Kolkata Wetlands,Cormorant,19.0,5.645269
4,2025-01-01,East Kolkata Wetlands,Ibis,8.0,4.971061


In [3]:
birds.info()
birds.describe()

<class 'pandas.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               144 non-null    datetime64[us]
 1   site               144 non-null    str           
 2   species            144 non-null    str           
 3   count              132 non-null    float64       
 4   observation_hours  144 non-null    float64       
dtypes: datetime64[us](1), float64(2), str(2)
memory usage: 5.8 KB


,date,count,observation_hours
count,144,132.000000,144.000000
mean,2025-03-17 04:00:00,17.280303,5.112726
min,2025-01-01 00:00:00,3.000000,2.033133
25%,2025-02-01 00:00:00,10.000000,3.663532
50%,2025-03-16 12:00:00,16.000000,5.025827
75%,2025-05-01 00:00:00,23.000000,6.680057
max,2025-06-01 00:00:00,43.000000,7.913903
std,NaN,8.816461,1.712267


In [4]:
birds['count'].unique()
birds[birds['count'].isna()]
birds.fillna(birds['count'].mean(), inplace=True)

,date,site,species,count,observation_hours
0,2025-01-01,East Kolkata Wetlands,Egret,15.0,2.935967
1,2025-01-01,East Kolkata Wetlands,Kingfisher,6.0,3.090950
2,2025-01-01,East Kolkata Wetlands,Heron,9.0,3.198043
3,2025-01-01,East Kolkata Wetlands,Cormorant,19.0,5.645269
4,2025-01-01,East Kolkata Wetlands,Ibis,8.0,4.971061
...,...,...,...,...,...
139,2025-06-01,Santragachi,Kingfisher,6.0,4.713308
140,2025-06-01,Santragachi,Heron,10.0,3.058322
141,2025-06-01,Santragachi,Cormorant,16.0,4.174363
142,2025-06-01,Santragachi,Ibis,21.0,5.981224


In [5]:
birds['month'] = birds['date'].dt.month
birds['day_name'] = birds['date'].dt.day_name()
birds['density'] = birds['count'] / birds['observation_hours']
birds.head()

,date,site,species,count,observation_hours,month,day_name,density
0,2025-01-01,East Kolkata Wetlands,Egret,15.0,2.935967,1,Wednesday,5.109049
1,2025-01-01,East Kolkata Wetlands,Kingfisher,6.0,3.090950,1,Wednesday,1.941151
2,2025-01-01,East Kolkata Wetlands,Heron,9.0,3.198043,1,Wednesday,2.814221
3,2025-01-01,East Kolkata Wetlands,Cormorant,19.0,5.645269,1,Wednesday,3.365650
4,2025-01-01,East Kolkata Wetlands,Ibis,8.0,4.971061,1,Wednesday,1.609314


In [6]:
birds[birds['count']>birds['count'].mean()]
birds.query('species == "Egret" | species =="Kingfisher" | species == "Heron" | species == "Cormorant" | species == "Ibis"').where(birds['count'] > birds['count'].mean()).dropna()
birds.query('species == "Kingfisher"').where(birds['site'] == 'Nalban').dropna().sort_values(by='date',ascending=False)

,date,site,species,count,observation_hours,month,day_name,density
133,2025-06-01,Nalban,Kingfisher,29.0,6.179905,6.0,Sunday,4.692629
109,2025-05-01,Nalban,Kingfisher,31.0,2.610695,5.0,Thursday,11.874233
85,2025-04-01,Nalban,Kingfisher,14.0,4.258780,4.0,Tuesday,3.287326
61,2025-03-01,Nalban,Kingfisher,3.0,7.069252,3.0,Saturday,0.424373
37,2025-02-01,Nalban,Kingfisher,26.0,5.945677,2.0,Saturday,4.372925
13,2025-01-01,Nalban,Kingfisher,16.0,7.323276,1.0,Wednesday,2.184814


In [7]:
birds.where(birds['observation_hours']>=birds['observation_hours'].quantile(0.9)).dropna()
birds['observation_hours'].quantile([0.25,0.5,0.75,0.9])

0.25    3.663532
0.50    5.025827
0.75    6.680057
0.90    7.448336
Name: observation_hours, dtype: float64

In [8]:
birds.groupby(['site','species'])['count'].agg(['min','max','mean','sum']).sort_values(by=['site','sum'] ,ascending=[True,False] )
birds.groupby('species')['count'].agg(['max','min','std'])
birds.groupby(['site','month'])['count'].agg(['sum']).sort_values(by=['site','sum'],ascending=[True,False])

sum
site                  month            
East Kolkata Wetlands 5      105.280303
                      6      101.000000
                      3       91.000000
                      4       87.560606
                      2       83.280303
                      1       63.000000
Nalban                6      161.000000
                      4      126.000000
                      5      105.000000
                      2      102.280303
                      3       98.280303
                      1       84.280303
Santragachi           4      130.280303
                      2      129.280303
                      5      108.000000
                      3       92.280303
                      1       92.000000
                      6       68.000000
Sundarban Fringe      1      134.000000
                      2      111.000000
                      3      111.000000
                      6      104.280303
                      4      101.280303
                      5       99.000000

In [9]:
birds_pivot = pd.pivot_table(birds, index='site', columns='species',values='count',aggfunc='sum')
birds_pivot

species,Cormorant,Egret,Heron,Ibis,Kingfisher,Pied Hornbill
site,,,,,,
East Kolkata Wetlands,127.560606,68.000000,76.280303,87.000000,62.280303,110.000000
Nalban,119.280303,95.000000,101.280303,123.000000,119.000000,119.280303
Santragachi,98.000000,85.280303,76.280303,175.280303,90.000000,95.000000
Sundarban Fringe,83.000000,111.280303,116.000000,94.000000,110.000000,146.280303


In [10]:
ovser_pivot = pd.pivot_table(birds,index='site',columns='species',values='observation_hours',aggfunc='mean')
ovser_pivot

species,Cormorant,Egret,Heron,Ibis,Kingfisher,Pied Hornbill
site,,,,,,
East Kolkata Wetlands,4.937580,5.210786,4.481279,3.789049,4.921066,6.052213
Nalban,6.111787,4.531874,5.274141,5.247168,5.564598,5.354473
Santragachi,6.066084,4.617324,4.717663,4.939191,5.527176,5.546335
Sundarban Fringe,4.175224,5.312795,5.317632,3.914753,4.800563,6.294670


### Day 2 — Water Quality & Pollution Analysis

In [11]:
import numpy as np
import pandas as pd

np.random.seed(24)

stations = ["Hooghly_A", "Hooghly_B", "Damodar_A", "Damodar_B", "Rupnarayan"]
months = pd.date_range("2025-01-01", "2025-08-01", freq="MS")

rows = []

for month in months:
    for station in stations:
        rows.append({
            "date": month,
            "station": station,
            "temperature": np.random.normal(27, 3),
            "dissolved_oxygen": np.random.normal(6, 1.2),
            "bod": np.random.normal(4, 1.5),
            "nitrate": np.random.normal(3.5, 1.5),
            "sampling_depth": np.random.uniform(0.5, 3.0)
        })

water = pd.DataFrame(rows)

# Introduce realistic missing measurements
missing_idx = np.random.choice(
    water.index,
    size=10,
    replace=False
)

water.loc[missing_idx, "dissolved_oxygen"] = np.nan
water.loc[
    np.random.choice(water.index, 6, replace=False),
    "nitrate"
] = np.nan

water.head()

,date,station,temperature,dissolved_oxygen,bod,nitrate,sampling_depth
0,2025-01-01,Hooghly_A,30.987637,5.075960,3.525579,2.013784,1.301298
1,2025-01-01,Hooghly_B,31.229921,NaN,4.119309,4.899379,1.118234
2,2025-01-01,Damodar_A,29.036414,8.267127,5.442308,NaN,2.606949
3,2025-01-01,Damodar_B,29.927131,NaN,5.889212,5.843651,1.069208
4,2025-01-01,Rupnarayan,31.178564,NaN,4.182503,5.311404,2.394495


In [12]:
water.info()
water.describe()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              40 non-null     datetime64[us]
 1   station           40 non-null     str           
 2   temperature       40 non-null     float64       
 3   dissolved_oxygen  30 non-null     float64       
 4   bod               40 non-null     float64       
 5   nitrate           34 non-null     float64       
 6   sampling_depth    40 non-null     float64       
dtypes: datetime64[us](1), float64(5), str(1)
memory usage: 2.3 KB


,date,temperature,dissolved_oxygen,bod,nitrate,sampling_depth
count,40,40.000000,30.000000,40.000000,34.000000,40.000000
mean,2025-04-16 12:00:00,28.238427,6.026277,4.226584,3.561294,1.801342
min,2025-01-01 00:00:00,21.396059,3.741178,1.044567,-0.479793,0.529882
25%,2025-02-22 00:00:00,26.177599,5.292514,3.408088,2.217142,1.135982
50%,2025-04-16 00:00:00,28.753565,6.061229,4.304779,3.953237,1.813775
75%,2025-06-08 12:00:00,30.841142,6.837596,5.450827,4.871481,2.416247
max,2025-08-01 00:00:00,32.811971,8.377701,6.992887,6.839239,2.997878
std,NaN,2.937573,1.199131,1.493567,1.762295,0.772820


In [13]:
# finding null values and replace them with mean
water[water['dissolved_oxygen'].isna()]
water.fillna(value={
    'dissolved_oxygen': water['dissolved_oxygen'].mean(),
    'bod' : water['bod'].mean(),
    'nitrate' : water['nitrate'].mean()
},inplace=True)

,date,station,temperature,dissolved_oxygen,bod,nitrate,sampling_depth
0,2025-01-01,Hooghly_A,30.987637,5.075960,3.525579,2.013784,1.301298
1,2025-01-01,Hooghly_B,31.229921,6.026277,4.119309,4.899379,1.118234
2,2025-01-01,Damodar_A,29.036414,8.267127,5.442308,3.561294,2.606949
3,2025-01-01,Damodar_B,29.927131,6.026277,5.889212,5.843651,1.069208
4,2025-01-01,Rupnarayan,31.178564,6.026277,4.182503,5.311404,2.394495
5,2025-02-01,Hooghly_A,30.578626,5.998680,5.485477,3.991703,1.079243
6,2025-02-01,Hooghly_B,32.059749,4.408844,6.143476,0.365969,2.853548
7,2025-02-01,Damodar_A,31.616836,5.292192,4.501427,4.477897,0.743812
8,2025-02-01,Damodar_B,30.792310,6.348042,1.044567,4.705859,1.858749
9,2025-02-01,Rupnarayan,25.555871,6.026277,4.905733,3.561294,2.138215


In [ ]:
# seperating months in a new column
water['month'] = water['date'].dt.month
water

In [ ]:
water.groupby('month')[['dissolved_oxygen','bod','nitrate']].agg(['min','max','mean'])

In [ ]:
water.groupby('month')[['dissolved_oxygen','bod','nitrate']].agg(['min','max','mean']).sort_values(by=[('dissolved_oxygen','min'),('bod','max'),('nitrate','max')],ascending=[True,False,False])


In [17]:
pol_score = water.groupby('month')[['dissolved_oxygen','bod','nitrate']].mean().sort_values(by=['dissolved_oxygen','bod','nitrate'],ascending=[True,False,False])
pol_score['score_pol'] = pol_score[['dissolved_oxygen','bod','nitrate']].to_numpy().sum(axis=1)
pol_score

,dissolved_oxygen,bod,nitrate,score_pol
month,,,,
3,5.186892,4.754373,4.009155,13.950419
2,5.614807,4.416136,3.420544,13.451487
6,5.908554,3.240017,3.309752,12.458324
4,6.075642,4.039748,2.956545,13.071935
8,6.211101,4.720302,3.725929,14.657333
1,6.284384,4.631782,4.325902,15.242068
5,6.370299,4.261740,4.120937,14.752977
7,6.558538,3.748570,2.621585,12.928693


In [18]:
pol_score[['dissolved_oxygen','bod']].to_numpy().sum(axis=1).reshape(-1,1)

array([[ 9.94126456],
       [10.03094298],
       [ 9.14857157],
       [10.11538992],
       [10.93140339],
       [10.91616575],
       [10.63203952],
       [10.30710759]])

In [19]:
pol_score = np.array(pol_score)
arr = np.sum(pol_score,axis=1)
arr = arr.reshape(-1,1)
pol_score = np.hstack((pol_score,arr))
pol_score

array([[ 5.18689194,  4.75437262,  4.00915493, 13.95041949, 27.90083898],
       [ 5.61480704,  4.41613595,  3.42054432, 13.45148731, 26.90297461],
       [ 5.90855418,  3.24001739,  3.30975195, 12.45832352, 24.91664704],
       [ 6.07564219,  4.03974773,  2.95654496, 13.07193488, 26.14386976],
       [ 6.21110102,  4.72030237,  3.7259293 , 14.6573327 , 29.3146654 ],
       [ 6.28438371,  4.63178204,  4.32590247, 15.24206822, 30.48413644],
       [ 6.37029904,  4.26174048,  4.12093701, 14.75297654, 29.50595307],
       [ 6.55853799,  3.7485696 ,  2.62158544, 12.92869303, 25.85738606]])

In [20]:
water.groupby('station')[['dissolved_oxygen','bod','nitrate']].apply(lambda df: pd.Series({
    'dis_ox_mean': df['dissolved_oxygen'].mean(),
    'bod_median' : df['bod'].median(),
    'nitrate_mean' : df['nitrate'].mean()
}))

,dis_ox_mean,bod_median,nitrate_mean
station,,,
Damodar_A,6.082432,3.591031,3.253785
Damodar_B,5.825224,4.197863,3.823220
Hooghly_A,6.129236,4.296038,2.969593
Hooghly_B,5.919697,4.254038,3.459793
Rupnarayan,6.174797,4.657338,4.300078


In [ ]:
# water.groupby('station')['bod'].apply(lambda per: (per > 5).mean()*100)
water.groupby('station').agg(
    mean_bod = ('bod', 'mean'),
    median_nitrate = ('nitrate', 'median'),
    std_diss_ox = ('dissolved_oxygen','std'),
    percentage_bod_above5 = ('bod', lambda per: (per>5).mean()*100)
).sort_values(by=['mean_bod','median_nitrate'],ascending=[False,False])

,mean_bod,median_nitrate,std_diss_ox,percentage_bod_above5
station,,,,
Hooghly_A,4.754002,3.461731,0.830046,37.5
Rupnarayan,4.542133,4.243051,0.796034,25.0
Hooghly_B,4.329145,3.738032,1.373928,37.5
Damodar_B,3.841177,3.982679,0.894348,25.0
Damodar_A,3.666460,3.561294,1.358965,12.5


In [ ]:
water['risk_class'] = np.where(
    (
        (water['bod']>5).astype(int)
        + (water['nitrate']>5).astype(int)
        + (water['dissolved_oxygen']<5).astype(int)
    )>=2,
    'high_risk',
    'normal'
)

water.groupby('station').agg(
    total_obs = ('risk_class','size'),
    risk_perc = ('risk_class',lambda x: (x == 'high_risk').sum())
)


,total_obs,risk_perc
station,,
Damodar_A,8,0
Damodar_B,8,2
Hooghly_A,8,0
Hooghly_B,8,2
Rupnarayan,8,2
